# RAD-DINO — Locked Final Test

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_classifier_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
from final_classifier_evaluation import *

DRY_RUN = True
RECOMPUTE_TEST_PREDICTIONS = False
ALLOW_UNVERIFIED_LEGACY_PREDICTIONS = False
TEST_BATCH_SIZE = 8
TEST_NUM_WORKERS = 4
DEVICE = "auto"
PATIENT_AGGREGATION = "mean"
TEST_CSV = PROJECT_ROOT / "data/processed/metadata/test.csv"
TEST_DATASET_MANIFEST = PROJECT_ROOT / "results/final_evaluation/test_dataset_manifest.json"
REGISTRY_PATH = PROJECT_ROOT / "configs/final_classifier_registry.json"

FAMILY = "raddino"
SUPPORTED_EXPERIMENT_IDS = ['raddino_04a_real_only', 'raddino_04b_real_synth']
LOCKED_MANIFEST_PATH = PROJECT_ROOT / "results/final_evaluation/finalists_manifest.json"
if LOCKED_MANIFEST_PATH.is_file():
    # Scientific-only validation: the finalist selection can be frozen and valid even while some
    # finalists (in this or other families) are still operationally blocked. This notebook only
    # needs its OWN family's operationally-ready members; it must not be blocked by others.
    scientific_lock = validate_locked_finalists_manifest(LOCKED_MANIFEST_PATH, require_operational_complete=False)
    print("scientific_selection_complete:", scientific_lock.get("scientific_selection_complete"),
          "| final_aggregation_complete:", scientific_lock.get("final_aggregation_complete"),
          "| operational_blockers:", scientific_lock.get("operational_blockers"))
    ready_ids = {x["experiment_id"] for x in scientific_lock["finalists"] if x.get("operationally_ready")}
    EXPERIMENT_IDS = [eid for eid in SUPPORTED_EXPERIMENT_IDS if eid in ready_ids]
else:
    EXPERIMENT_IDS = SUPPORTED_EXPERIMENT_IDS


In [ ]:
registry = {x["experiment_id"]: x for x in build_experiment_registry(REGISTRY_PATH)}
experiments = [registry[eid] for eid in EXPERIMENT_IDS]
for exp in experiments:
    exp["test_csv"] = "data/processed/metadata/test.csv"
    checked = validate_locked_test_configuration(exp, PROJECT_ROOT)
    canonical_paths = canonical_test_prediction_paths(exp)
    output_dir = (PROJECT_ROOT / canonical_paths["test_predictions_path"]).parent
    print({
        "experiment": exp["experiment_id"], "checkpoint": exp["checkpoint_path"],
        "checkpoint_signature": checked["checkpoint_signature"], "validation_threshold": checked["validation_threshold"],
        "threshold_method": checked["threshold_method"], "test_n": len(pd.read_csv(TEST_CSV)),
        "cache_available": (output_dir / "test_predictions.csv").is_file(), "output": str(output_dir),
        "device": DEVICE, "dry_run": DRY_RUN,
    })
if DRY_RUN:
    print("DRY_RUN: nessun modello caricato, nessuna GPU allocata, nessun file scritto.")

In [ ]:
if not DRY_RUN:
    import torch
    from medfoundation_utils import build_medfoundation_model, make_medfoundation_dataloader, predict_probs, resolve_normalization_medfoundation
    device = torch.device("cuda" if DEVICE == "auto" and torch.cuda.is_available() else ("cpu" if DEVICE == "auto" else DEVICE))
    test_df = pd.read_csv(TEST_CSV); test_df["resolved_path"] = test_df["processed_path"].map(lambda x: str(PROJECT_ROOT / x))
    for exp in experiments:
        canonical_paths = canonical_test_prediction_paths(exp)
        pred_path = PROJECT_ROOT / canonical_paths["test_predictions_path"]
        manifest_path = PROJECT_ROOT / canonical_paths["test_predictions_manifest_path"]
        output_dir = pred_path.parent
        expected_cache = {"experiment_id": exp["experiment_id"], "checkpoint_signature": content_signature(PROJECT_ROOT / exp["checkpoint_path"]), "validation_metrics_signature": content_signature(PROJECT_ROOT / exp["validation_metrics_path"]), "validation_threshold": exp["validation_threshold"], "threshold_method": exp["validation_threshold_method"], "test_dataset_manifest_signature": content_signature(TEST_DATASET_MANIFEST), "patient_ids_hash": patient_ids_hash(test_df.patient_id), "preprocessing": {"resolution": 512, "grayscale_to_rgb": True}, "model_config": {"model": "microsoft/rad-dino"}, "pipeline_schema_version": 1, "provenance_level": "verified_recomputed"}
        cache = prediction_cache_status(manifest_path, expected_cache, pred_path); print(cache["status"], cache["incompatible_keys"])
        if cache["status"] == "CACHE_VALID" and not RECOMPUTE_TEST_PREDICTIONS: continue
        if pred_path.exists() and not RECOMPUTE_TEST_PREDICTIONS: raise RuntimeError(f"CACHE_INCOMPATIBLE: {cache['incompatible_keys']}")
        model, processor, _ = build_medfoundation_model("microsoft/rad-dino", num_classes=1)
        state = unwrap_checkpoint_state_dict(torch.load(PROJECT_ROOT / exp["checkpoint_path"], map_location=device))
        missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
        mismatch = checkpoint_key_mismatch(missing_keys, unexpected_keys)
        if mismatch["unexplained_missing"] or mismatch["unexplained_unexpected"]: raise RuntimeError(f"Checkpoint RAD-DINO incompatibile: {mismatch}")
        model.to(device).eval(); mean, std, img_size = resolve_normalization_medfoundation(processor)
        loader = make_medfoundation_dataloader(test_df, "resolved_path", "label", mean, std, img_size, TEST_BATCH_SIZE, False, False, TEST_NUM_WORKERS, 42, False)
        y_true, y_score = predict_probs(model, loader, device)
        raw = test_df[["patient_id", "image_id", "processed_path"]].rename(columns={"processed_path": "path"}); raw["y_true"], raw["y_score"] = y_true.astype(int), y_score
        standardized = standardize_prediction_dataframe(raw, experiment=exp, threshold=exp["validation_threshold"], threshold_method=exp["validation_threshold_method"])
        output_dir.mkdir(parents=True, exist_ok=True); standardized.to_csv(pred_path, index=False)
        metrics = compute_binary_metrics(standardized.y_true, standardized.y_score, exp["validation_threshold"])
        (output_dir / "test_metrics.json").write_text(strict_json_dumps(metrics, indent=2) + "\n"); pd.DataFrame([metrics | {"confusion_matrix": json.dumps(metrics["confusion_matrix"])}]).to_csv(output_dir / "test_metrics.csv", index=False); pd.DataFrame(metrics["confusion_matrix"]).to_csv(output_dir / "confusion_matrix.csv", index=False)
        manifest = {**expected_cache, "preprocessing_details": {"resolution": img_size, "grayscale_to_rgb": True, "mean": mean, "std": std}, "n_patients": len(standardized), "test_used_for_selection": False}
        write_prediction_manifest(manifest_path, manifest, pred_path)